# The tree solver, measured against direct summation

Barnes-Hut replaces distant groups of particles with a single term each, which
turns the cost from `N^2` into something close to `N log N` and introduces an
error the opening angle controls. Direct summation computes every pair and is
the reference: it is not deleted once faster methods exist, and every
approximation in this project is measured against it.

`compute_accelerations` is what makes that comparison one line from Python. It
evaluates the forces on a set of particles once, in place, with whichever solver
the configuration names, and returns the work it took.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import orrery

seed = 20260812


def solver_configuration(kind, opening_angle=0.5, softening=0.02, quadrupole=False):
    configuration = orrery.Configuration()
    configuration.solver.kind = kind
    configuration.solver.opening_angle = opening_angle
    configuration.solver.softening = softening
    configuration.solver.quadrupole = quadrupole
    return configuration


def accelerations(particles, configuration):
    """The accelerations of a copy of `particles`, and the work it took."""
    data = particles.copy()
    count = orrery.compute_accelerations(configuration, data)
    return orrery.stacked(data, "acceleration"), count

## Error against opening angle

The opening angle decides when a cell is far enough away to be replaced by its
centre of mass. Closing it opens more cells, costs more interactions, and gets
closer to the exact answer. The error is quoted relative to the root mean square
acceleration, so it is a fraction rather than a number in units nobody stated.

In [ ]:
particles = orrery.plummer_sphere(orrery.PlummerParameters(count=8192), seed=seed)

exact, direct_count = accelerations(particles, solver_configuration(orrery.SolverKind.direct))
scale = np.sqrt(np.mean(np.sum(exact**2, axis=1)))

angles = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
rms, worst, interactions = [], [], []

for angle in angles:
    approximate, count = accelerations(
        particles, solver_configuration(orrery.SolverKind.barnes_hut, angle)
    )
    residual = np.sqrt(np.sum((approximate - exact) ** 2, axis=1))
    rms.append(float(np.sqrt(np.mean(residual**2))) / scale)
    worst.append(float(residual.max()) / scale)
    interactions.append(count.particle_particle + count.particle_cell)

print(f"{'angle':>6} {'rms error':>12} {'worst':>12} {'interactions':>14}")
for angle, r, w, n in zip(angles, rms, worst, interactions):
    print(f"{angle:6.2f} {r:12.3e} {w:12.3e} {n:14,}")

print()
print(f"direct summation: {direct_count.particle_particle:,} interactions, error zero by definition")

# Closing the angle has to buy accuracy and has to cost work, or the parameter
# does not mean what the algorithm says it means.
assert rms[0] < rms[-1]
assert interactions[0] > interactions[-1]

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].semilogy(angles, rms, marker="o", label="root mean square")
axes[0].semilogy(angles, worst, marker="s", label="worst particle")
axes[0].axvline(0.5, color="grey", linestyle="--", linewidth=1)
axes[0].set_xlabel("opening angle")
axes[0].set_ylabel("relative acceleration error")
axes[0].set_title("accuracy against the opening angle")
axes[0].legend()
axes[0].grid(True, which="both", alpha=0.3)

# The curve that decides a default: error against the work it costs. The dashed
# line is direct summation, which is exact and to the right of everything.
axes[1].loglog(interactions, rms, marker="o")
for angle, n, r in zip(angles, interactions, rms):
    axes[1].annotate(f"{angle:g}", (n, r), textcoords="offset points", xytext=(4, 4), fontsize=8)
axes[1].axvline(direct_count.particle_particle, color="grey", linestyle="--", linewidth=1)
axes[1].text(
    direct_count.particle_particle, max(rms), " direct", fontsize=8, va="top", color="grey"
)
axes[1].set_xlabel("interactions computed")
axes[1].set_ylabel("root mean square relative error")
axes[1].set_title("error against cost")
axes[1].grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

## Where the work goes

The interaction counter says something the timings cannot. It reports how many
particle-particle and particle-cell terms the solver computed, so the saving is
a property of the algorithm rather than of the machine it ran on.

In [ ]:
counts = []
sizes = [1024, 2048, 4096, 8192, 16384]

for count_of_particles in sizes:
    sample = orrery.plummer_sphere(
        orrery.PlummerParameters(count=count_of_particles), seed=seed
    )
    _, tree = accelerations(sample, solver_configuration(orrery.SolverKind.barnes_hut))
    _, direct = accelerations(sample, solver_configuration(orrery.SolverKind.direct))
    counts.append((tree.particle_particle + tree.particle_cell, direct.particle_particle))

tree_counts = np.array([c[0] for c in counts], dtype=float)
direct_counts = np.array([c[1] for c in counts], dtype=float)

print(f"{'N':>7} {'tree':>14} {'direct':>16} {'ratio':>8} {'per particle':>14}")
for n, tree_n, direct_n in zip(sizes, tree_counts, direct_counts):
    print(f"{n:7} {tree_n:14,.0f} {direct_n:16,.0f} {direct_n / tree_n:8.1f}x {tree_n / n:14.0f}")

figure, axes = plt.subplots(figsize=(7, 4.5))
axes.loglog(sizes, direct_counts, marker="s", label="direct, N^2")
axes.loglog(sizes, tree_counts, marker="o", label="tree")
axes.set_xlabel("particles")
axes.set_ylabel("interactions per force evaluation")
axes.set_title("the cost the algorithms actually pay")
axes.legend()
axes.grid(True, which="both", alpha=0.3)
plt.show()

# The saving grows with N, which is the whole claim of a hierarchical method.
assert (direct_counts / tree_counts)[-1] > (direct_counts / tree_counts)[0]

## The invariant the tree gives up

Direct summation conserves linear momentum to round-off because it computes each
pair from both ends, so the two equal and opposite contributions cancel exactly.
A tree does not: particle `i` may see `j` through a cell while `j` sees `i`
directly, and the pair of forces is then not equal and opposite.

The size of that violation is measured here rather than assumed to be zero, and
it falls as the angle closes.

In [ ]:
masses = particles.mass


def momentum_violation(configuration):
    field, _ = accelerations(particles, configuration)
    total = np.sum(field * masses[:, None], axis=0)
    return float(np.linalg.norm(total)) / float(np.sum(np.abs(field) * masses[:, None]))


direct_violation = momentum_violation(solver_configuration(orrery.SolverKind.direct))
tree_violations = [
    momentum_violation(solver_configuration(orrery.SolverKind.barnes_hut, angle))
    for angle in angles
]

print(f"direct summation: {direct_violation:.3e}, which is round-off")
print()
print(f"{'angle':>6} {'momentum violation':>20}")
for angle, violation in zip(angles, tree_violations):
    print(f"{angle:6.2f} {violation:20.3e}")

assert direct_violation < 1e-12
assert tree_violations[0] < tree_violations[-1]
print()
print("the tree violates momentum conservation, and closing the angle reduces it")

## What was shown

- The tree's error against the direct reference falls as the opening angle
  closes, and the cost rises, which is the curve that justifies the default of
  0.5.
- The saving in interactions grows with the particle count, which is what a
  hierarchical method is for.
- The tree gives up exact momentum conservation, by an amount that is measured
  rather than assumed, and closing the angle reduces it.

The same comparisons, with timings on the machine the project was built for, are
in `docs/performance/barnes_hut.md`.